# Proyecto Final

## Parte 2. Generación y análisis de secuencias para un binder de patulina

**authors**: Barrientos-González Diana, Contreras-Rivera Jair Emiliano, González-López Leonardo Daniel

**date**: 22-11-2026

**materia**: Bioinformática



### Material de entrada



*   Archivo .pdb con el esqueleto proteico (backbone) de un binder contra patulina diseñado con RFdiffusion.


*   Archivo .txt con la lista de residuos involucrados en la unión del binder con  su ligando.


### Contexto
En esta actividad utilizarás ProteinMPNN para diseñar secuencias de aminoácidos compatibles con el
esqueleto estructural (backbone) de un binder contra patulina, previamente generado mediante RFdiffusion.


El objetivo es explorar cómo diferentes condiciones de diseño afectan las secuencias generadas y sus
puntuaciones. En particular, compararás el efecto de modificar la temperatura de muestreo y de conservar o
permitir la modificación de los residuos involucrados en la interacción con la patulina.

### Upload de los archivos y preview de su contenido

In [82]:
!pip install biopython
#libreria necesaria

In [83]:
import os
print(os.listdir("/content/sample_data/bioinfo"))#caragar archvios de forma rmanual y
# ver donde estan ubicados mis archivos

['binder_patulin_315.pdb', 'fixed_residues_binder_315.txt']


In [84]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)

structure = parser.get_structure(
    "binder",
    "/content/sample_data/bioinfo/binder_patulin_315.pdb"
    #RECONOCED MI ARCHIVO pdb
)

In [85]:
print("Número de modelos:", len(structure))
#checa todos tipos de cadenas o modelos que encuentre en nuestroa rchivo PDB de entrada
for chain in structure[0]:
    residues = [
        r for r in chain
        if r.id[0] == " "
    ]

    print(
        "Cadena:", chain.id,
        "| Número de residuos:", len(residues)
    )

Número de modelos: 1
Cadena: A | Número de residuos: 93
Cadena: B | Número de residuos: 0


In [86]:
ruta_txt = "/content/sample_data/bioinfo/fixed_residues_binder_315.txt"
#revisamos el segundoa rchivo a ver si imprime lo que deseeamos que encuebntre
with open(ruta_txt, "r") as archivo:
    contenido = archivo.read()

print(contenido)

Phe 9
Leu 12
Phe 13
Phe 40
Ser 53
Val 59
Lys 61



In [87]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure("binder", "/content/sample_data/bioinfo/binder_patulin_315.pdb")

for chain in structure[0]:
    print("Cadena:", chain.id)

    for residue in chain:
        print(
            residue.id,
            residue.resname
        )

Cadena: A
(' ', 1, ' ') ALA
(' ', 2, ' ') ALA
(' ', 3, ' ') ALA
(' ', 4, ' ') ALA
(' ', 5, ' ') ALA
(' ', 6, ' ') ALA
(' ', 7, ' ') ALA
(' ', 8, ' ') ALA
(' ', 9, ' ') PHE
(' ', 10, ' ') ALA
(' ', 11, ' ') ALA
(' ', 12, ' ') LEU
(' ', 13, ' ') PHE
(' ', 14, ' ') ALA
(' ', 15, ' ') ALA
(' ', 16, ' ') ALA
(' ', 17, ' ') ALA
(' ', 18, ' ') ALA
(' ', 19, ' ') ALA
(' ', 20, ' ') ALA
(' ', 21, ' ') ALA
(' ', 22, ' ') ALA
(' ', 23, ' ') ALA
(' ', 24, ' ') ALA
(' ', 25, ' ') ALA
(' ', 26, ' ') ALA
(' ', 27, ' ') ALA
(' ', 28, ' ') ALA
(' ', 29, ' ') ALA
(' ', 30, ' ') ALA
(' ', 31, ' ') ALA
(' ', 32, ' ') ALA
(' ', 33, ' ') ALA
(' ', 34, ' ') ALA
(' ', 35, ' ') ALA
(' ', 36, ' ') ALA
(' ', 37, ' ') ALA
(' ', 38, ' ') ALA
(' ', 39, ' ') ALA
(' ', 40, ' ') PHE
(' ', 41, ' ') ALA
(' ', 42, ' ') ALA
(' ', 43, ' ') ALA
(' ', 44, ' ') ALA
(' ', 45, ' ') ALA
(' ', 46, ' ') ALA
(' ', 47, ' ') ALA
(' ', 48, ' ') ALA
(' ', 49, ' ') ALA
(' ', 50, ' ') ALA
(' ', 51, ' ') ALA
(' ', 52, ' ') ALA
(' ', 53, '

**Contexto**

Esto nos dice que sólo tenemos una sola estructura tridimensional con 93 residuos

Encontró una cadena B pero con 0 residuos si nos vamos al archivo veremos que tiene una molécuña identificada como I8L  el cual es un idenfificador de uan molécula no estándar, por lo cual seguramente es una molécula muy pequeña, en contraste con la cadena A este tiene por código ALA que es un aminoácido de la proteína. Hipotetizamos que I8L puede ser patulina

Nuestro archivo de residuos de interés txt contiene las siguientes posiciones:

Phe 9, Leu 12, Phe 13, Phe 40, Ser 53, Val 59, Lys 61

Por lo cual estas posiciones entran dentro de mi estructura A de proteínas

Checando en PDB, I8L se encuentra como un ligando independiente (o aislado) y no es un polímero: https://www.rcsb.org/ligand/I8L?utm_source=chatgpt.com


### ProteinMPNN


## Parámetros de Diseño

### A. Temperatura de muestreo (Sampling Temperature, T)

La temperatura controla la diversidad de las secuencias generadas por ProteinMPNN. Para este ejercicio
utilizarás tres valores:


*   T = 0.01: muestreo muy conservador.Favorece aminoácidos con mayor probabilidad predicha por el
modelo y tiende a generar secuencias menos diversas.



*   T = 0.1: muestreo de baja temperatura. Permite cierta variabilidad manteniendo una exploración
relativamente conservadora.



*    T = 1: muestreo de mayor temperatura. Permite una exploración más amplia del espacio de secuencias y puede generar mayor diversidad.


### B. Residuos fijados (fixed positions)

Realizarás dos estrategias de diseño:



*   Con residuos fijados: los residuos indicados deberán conservar su
identidad aminoacídica original.
ProteinMPNN podrá modificar el resto de las posiciones del binder.+



*   Sin residuos fijados: ProteinMPNN podrá rediseñar todas las posiciones de la secuencia del binder,
incluidos los residuos involucrados en la interacción con la patulina.

**Nota**: Este código fue copiado desde, los tres primeros chunks quedaaron idénticos mientras que los últimos fueron adaptados según  lo requerido por la actividad por nosotros https://colab.research.google.com/github/dauparas/ProteinMPNN/blob/main/colab_notebooks/quickdemo.ipynb?utm_source=chatgpt.com#scrollTo=yjnUkQuZX-de



In [88]:
#@title Clone github repo
!git clone https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN
import json, time, os, sys, glob

if not os.path.isdir("ProteinMPNN"):
  os.system("git clone -q https://github.com/dauparas/ProteinMPNN.git")
sys.path.append('/content/ProteinMPNN')



fatal: destination path '/content/ProteinMPNN' already exists and is not an empty directory.


In [89]:
import os, sys, subprocess

print("Directorio actual:", os.getcwd())
print("¿Existe /content?", os.path.exists('/content'))
print("¿Existe /content/ProteinMPNN?", os.path.exists('/content/ProteinMPNN'))

if os.path.exists('/content/ProteinMPNN'):
    print("Contenido:", os.listdir('/content/ProteinMPNN'))

print("¿Existe ./ProteinMPNN?", os.path.exists('ProteinMPNN'))

Directorio actual: /content
¿Existe /content? True
¿Existe /content/ProteinMPNN? True
Contenido: ['colab_notebooks', 'training', 'inputs', 'README.md', 'ca_model_weights', 'LICENSE', 'soluble_model_weights', 'protein_mpnn_utils.py', 'protein_mpnn_run.py', 'helper_scripts', 'vanilla_model_weights', '__pycache__', 'examples', 'outputs', '.git', '.gitignore', 'ProteinMPNN']
¿Existe ./ProteinMPNN? True


In [90]:
#@title Setup Model
import sys
import os

# Agregamos tanto la raíz como el subdirectorio de scripts auxiliares para asegurar todas las importaciones
paths_to_add = [
    "/content/ProteinMPNN",
    "/content/ProteinMPNN/helper_scripts"
]
for folder in paths_to_add:
    abs_folder = os.path.abspath(folder)
    if abs_folder not in sys.path:
        sys.path.insert(0, abs_folder)

import matplotlib.pyplot as plt
import shutil
import warnings
import numpy as np
import torch
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split, Subset
import copy
import torch.nn as nn
import torch.nn.functional as F
import random
import os.path

# Ahora que los paths están configurados con máxima prioridad, importamos sin errores
from protein_mpnn_utils import loss_nll, loss_smoothed, gather_edges, gather_nodes, gather_nodes_t, cat_neighbors_nodes, _scores, _S_to_seq, tied_featurize, parse_PDB
from protein_mpnn_utils import StructureDataset, StructureDatasetPDB, ProteinMPNN

device = torch.device("cuda:0" if (torch.cuda.is_available()) else "cpu")
#v_48_010=version with 48 edges 0.10A noise
model_name = "v_48_020" #@param ["v_48_002", "v_48_010", "v_48_020", "v_48_030"]

backbone_noise=0.00               # Standard deviation of Gaussian noise to add to backbone atoms

path_to_model_weights='/content/ProteinMPNN/vanilla_model_weights'
hidden_dim = 128
num_layers = 3
model_folder_path = path_to_model_weights
if model_folder_path[-1] != '/':
    model_folder_path = model_folder_path + '/'
checkpoint_path = model_folder_path + f'{model_name}.pt'

checkpoint = torch.load(checkpoint_path, map_location=device)
print('Number of edges:', checkpoint['num_edges'])
noise_level_print = checkpoint['noise_level']
print(f'Training noise level: {noise_level_print}A')
model = ProteinMPNN(num_letters=21, node_features=hidden_dim, edge_features=hidden_dim, hidden_dim=hidden_dim, num_encoder_layers=num_layers, num_decoder_layers=num_layers, augment_eps=backbone_noise, k_neighbors=checkpoint['num_edges'])
model.to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Model loaded")

Number of edges: 48
Training noise level: 0.2A
Model loaded


In [91]:
#@title Helper functions
def make_tied_positions_for_homomers(pdb_dict_list):
    my_dict = {}
    for result in pdb_dict_list:
        all_chain_list = sorted([item[-1:] for item in list(result) if item[:9]=='seq_chain']) #A, B, C, ...
        tied_positions_list = []
        chain_length = len(result[f"seq_chain_{all_chain_list[0]}"])
        for i in range(1,chain_length+1):
            temp_dict = {}
            for j, chain in enumerate(all_chain_list):
                temp_dict[chain] = [i] #needs to be a list
            tied_positions_list.append(temp_dict)
        my_dict[result['name']] = tied_positions_list
    return my_dict

### 3. Generación de secuencias con ProteinMPNN

In [92]:
import re
import os
import sys
from google.colab import files
import numpy as np

# Agregamos tanto la raíz como el subdirectorio de scripts auxiliares
paths_to_add = [
    "/content/ProteinMPNN",
    "/content/ProteinMPNN/helper_scripts"
]
for folder in paths_to_add:
    abs_folder = os.path.abspath(folder)
    if abs_folder not in sys.path:
        sys.path.insert(0, abs_folder)

# Importamos las clases y métodos auxiliares necesarios que sí existen en el archivo
from protein_mpnn_utils import parse_PDB, StructureDatasetPDB

# Usamos la ruta local directa de tu archivo de proteína
pdb_path = "/content/sample_data/bioinfo/binder_patulin_315.pdb"

homomer = True
designed_chain = "A" #@param {type:"string"}
fixed_chain = "" #@param {type:"string"}

if designed_chain == "":
  designed_chain_list = []
else:
  designed_chain_list = re.sub("[^A-Za-z]+",",", designed_chain).split(",")

if fixed_chain == "":
  fixed_chain_list = []
else:
  fixed_chain_list = re.sub("[^A-Za-z]+",",", fixed_chain).split(",")

chain_list = list(set(designed_chain_list + fixed_chain_list))

#@markdown ### Design Options
# 1. Configurado para generar 100 secuencias por corrida
num_seqs = 100 #@param {type:"integer"}
num_seq_per_target = num_seqs

# 2. Temperatura de muestreo (Prueba cambiando entre "0.01", "0.1" y "1.0")
sampling_temp = "0.01" #@param ["0.01", "0.1", "1.0"] {type:"string"}

save_score=0                      # 0 for False, 1 for True; save score=-log_prob to npy files
save_probs=0                      # 0 for False, 1 for True; save MPNN predicted probabilites per position
score_only=0                      # 0 for False, 1 for True; score input backbone-sequence pairs
conditional_probs_only=0          # 0 for False, 1 for True; output conditional probabilities p(s_i given the rest of the sequence and backbone)
conditional_probs_only_backbone=0 # 0 for False, 1 for True; if true output conditional probabilities p(s_i given backbone)

batch_size=1                      # Batch size; can set higher for titan, quadro GPUs, reduce this if running out of GPU memory
max_length=20000                  # Max sequence length

out_folder='.'                    # Path to a folder to output sequences, e.g. /home/out/
jsonl_path=''                     # Path to a folder with parsed pdb into jsonl
omit_AAs='X'                      # Specify which amino acids should be omitted in the generated sequence, e.g. 'AC' would omit alanine and cystine.

pssm_multi=0.0                    # A value between [0.0, 1.0], 0.0 means do not use pssm, 1.0 ignore MPNN predictions
pssm_threshold=0.0                # A value between -inf + inf to restric per position AAs
pssm_log_odds_flag=0               # 0 for False, 1 for True
pssm_bias_flag=0                   # 0 for False, 1 for True

##############################################################

folder_for_outputs = out_folder

NUM_BATCHES = num_seq_per_target//batch_size
BATCH_COPIES = batch_size
temperatures = [float(item) for item in sampling_temp.split()]
omit_AAs_list = omit_AAs
alphabet = 'ACDEFGHIKLMNPQRSTVWYX'

omit_AAs_np = np.array([AA in omit_AAs_list for AA in alphabet]).astype(np.float32)

chain_id_dict = None
fixed_positions_dict = None
pssm_dict = None
omit_AA_dict = None
bias_AA_dict = None
tied_positions_dict = None
bias_by_res_dict = None
bias_AAs_np = np.zeros(len(alphabet))

###############################################################
pdb_dict_list = parse_PDB(pdb_path, input_chain_list=chain_list)
dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=max_length)

chain_id_dict = {}
chain_id_dict[pdb_dict_list[0]['name']]= (designed_chain_list, fixed_chain_list)

print(chain_id_dict)
for chain in chain_list:
  l = len(pdb_dict_list[0][f"seq_chain_{chain}"])
  print(f"Length of chain {chain} is {l}")

if homomer:
  # Aquí se mandará a llamar la función local que ya tienes definida en la celda yjnUkQuZX-de
  tied_positions_dict = make_tied_positions_for_homomers(pdb_dict_list)
else:
  tied_positions_dict = None

{'binder_patulin_315': (['A'], [])}
Length of chain A is 93


In [93]:
import os

# Listamos los archivos en el directorio actual para ver qué se ha guardado
archivos_en_content = os.listdir('.')
print("Archivos y carpetas en /content:")
for archivo in archivos_en_content:
    # Evitamos mostrar carpetas ocultas o de sistema para mayor claridad
    if not archivo.startswith('.'):
        print(f" - {archivo}")

Archivos y carpetas en /content:
 - secuencias_diseñadas.fasta
 - secuencias_temp_0.1_sin_fijos.fasta
 - secuencias_temp_1.0_sin_fijos.fasta
 - binder_patulin_315.pdb
 - secuencias_temp_0.01_con_fijos.fasta
 - tmp.pdb
 - secuencias_temp_0.1_con_fijos.fasta
 - score_top_bottom.csv
 - secuencias_temp_1.0_con_fijos.fasta
 - secuencias_temp_0.01_sin_fijos.fasta
 - ProteinMPNN
 - sample_data


In [94]:
#@title RUN
with torch.no_grad():
  print('Generating sequences...')
  for ix, protein in enumerate(dataset_valid):
    score_list = []
    all_probs_list = []
    all_log_probs_list = []
    S_sample_list = []
    batch_clones = [copy.deepcopy(protein) for i in range(BATCH_COPIES)]
    X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, visible_list_list, masked_list_list, masked_chain_length_list_list, chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, tied_pos_list_of_lists_list, pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, tied_beta = tied_featurize(batch_clones, device, chain_id_dict, fixed_positions_dict, omit_AA_dict, tied_positions_dict, pssm_dict, bias_by_res_dict)
    pssm_log_odds_mask = (pssm_log_odds_all > pssm_threshold).float() #1.0 for true, 0.0 for false
    name_ = batch_clones[0]['name']

    randn_1 = torch.randn(chain_M.shape, device=X.device)
    log_probs = model(X, S, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_1)
    mask_for_loss = mask*chain_M*chain_M_pos
    scores = _scores(S, log_probs, mask_for_loss)
    native_score = scores.cpu().data.numpy()

    for temp in temperatures:
        for j in range(NUM_BATCHES):
            randn_2 = torch.randn(chain_M.shape, device=X.device)
            if tied_positions_dict == None:
                sample_dict = model.sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), bias_by_res=bias_by_res_all)
                S_sample = sample_dict["S"]
            else:
                sample_dict = model.tied_sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), tied_pos=tied_pos_list_of_lists_list[0], tied_beta=tied_beta, bias_by_res=bias_by_res_all)
            # Compute scores
                S_sample = sample_dict["S"]
            log_probs = model(X, S_sample, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_2, use_input_decoding_order=True, decoding_order=sample_dict["decoding_order"])
            mask_for_loss = mask*chain_M*chain_M_pos
            scores = _scores(S_sample, log_probs, mask_for_loss)
            scores = scores.cpu().data.numpy()
            all_probs_list.append(sample_dict["probs"].cpu().data.numpy())
            all_log_probs_list.append(log_probs.cpu().data.numpy())
            S_sample_list.append(S_sample.cpu().data.numpy())
            for b_ix in range(BATCH_COPIES):
                masked_chain_length_list = masked_chain_length_list_list[b_ix]
                masked_list = masked_list_list[b_ix]
                seq_recovery_rate = torch.sum(torch.sum(torch.nn.functional.one_hot(S[b_ix], 21)*torch.nn.functional.one_hot(S_sample[b_ix], 21),axis=-1)*mask_for_loss[b_ix])/torch.sum(mask_for_loss[b_ix])
                seq = _S_to_seq(S_sample[b_ix], chain_M[b_ix])
                score = scores[b_ix]
                score_list.append(score)
                native_seq = _S_to_seq(S[b_ix], chain_M[b_ix])
                if b_ix == 0 and j==0 and temp==temperatures[0]:
                    start = 0
                    end = 0
                    list_of_AAs = []
                    for mask_l in masked_chain_length_list:
                        end += mask_l
                        list_of_AAs.append(native_seq[start:end])
                        start = end
                    native_seq = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
                    l0 = 0
                    for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
                        l0 += mc_length
                        native_seq = native_seq[:l0] + '/' + native_seq[l0:]
                        l0 += 1
                    sorted_masked_chain_letters = np.argsort(masked_list_list[0])
                    print_masked_chains = [masked_list_list[0][i] for i in sorted_masked_chain_letters]
                    sorted_visible_chain_letters = np.argsort(visible_list_list[0])
                    print_visible_chains = [visible_list_list[0][i] for i in sorted_visible_chain_letters]
                    native_score_print = np.format_float_positional(np.float32(native_score.mean()), unique=False, precision=4)
                    line = '>{}, score={}, fixed_chains={}, designed_chains={}, model_name={}\n{}\n'.format(name_, native_score_print, print_visible_chains, print_masked_chains, model_name, native_seq)
                    print(line.rstrip())
                start = 0
                end = 0
                list_of_AAs = []
                for mask_l in masked_chain_length_list:
                    end += mask_l
                    list_of_AAs.append(seq[start:end])
                    start = end

                seq = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
                l0 = 0
                for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
                    l0 += mc_length
                    seq = seq[:l0] + '/' + seq[l0:]
                    l0 += 1
                score_print = np.format_float_positional(np.float32(score), unique=False, precision=4)
                seq_rec_print = np.format_float_positional(np.float32(seq_recovery_rate.detach().cpu().numpy()), unique=False, precision=4)
                line = '>T={}, sample={}, score={}, seq_recovery={}\n{}\n'.format(temp,b_ix,score_print,seq_rec_print,seq)
                print(line.rstrip())


all_probs_concat = np.concatenate(all_probs_list)
all_log_probs_concat = np.concatenate(all_log_probs_list)
S_sample_concat = np.concatenate(S_sample_list)

Generating sequences...
>binder_patulin_315, score=1.6560, fixed_chains=[], designed_chains=['A'], model_name=v_48_020
AAAAAAAAFAALFAAAAAAAAAAAAAAAAAAAAAAAAAAFAAAAAAAAAAAASAAAAAVAKAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
>T=0.01, sample=0, score=0.9492, seq_recovery=0.2903
ALEAAVAATAAFLGTPGLRAGIEAGAEELGIAPEEAIRAIGELAADPATRPGFEALTGEGLARTEGLGPEERAAALVELGRALVRWAAALRAA
>T=0.01, sample=0, score=1.0135, seq_recovery=0.2258
AEAAAVARVRAALSTPGRRAGIERGAEELGISPEEAIERIGRLAADPATRPLFEALTGEGLAKAVGLDPARRAEVEVELGVALVRRAVELEAA
>T=0.01, sample=0, score=0.9292, seq_recovery=0.3548
AEAATIAAVRAALGTPGLAAGIEAGAAALGIAPETAVERIGRLAADPATRPAVVALTGAGLAAAAGLDPAARAEALVALGRALVRWAAEVEAA
>T=0.01, sample=0, score=0.9914, seq_recovery=0.3656
SVAAAIAAVRALLGTPAFRAGVEAGAAALGIDTETVIERIGRLAADPATRPLVVALTGAHLAAAVGLDPASRAAAAVEAGRALAALAAALEAA
>T=0.01, sample=0, score=0.8948, seq_recovery=0.3763
ALEAAIAAVRAALGQPAAAAGIAAGAAELGIAPEEAVARIGELAADPATRPLVEALTGAGLARAAGLPPAERAAALVELGRALVRHAAALAAA
>T=0.01, sample=0, score=0.9271, seq_recovery=0.3656

In [95]:
# Guardamos las secuencias generadas en formato FASTA
fasta_filename = "/content/secuencias_diseñadas.fasta"

# Definimos tag_fijos según el estado actual de las variables para evitar un NameError
tag_fijos = "con_fijos" if (locals().get('fixed_positions_dict') is not None and fixed_positions_dict is not None) else "sin_fijos"

with open(fasta_filename, "w") as fasta_file:
    index_seq = 0
    for temp in temperatures:
        for j in range(NUM_BATCHES):
            for b_ix in range(BATCH_COPIES):
                # Obtenemos la secuencia de aminoácidos actual
                seq_string = _S_to_seq(S_sample_list[index_seq][b_ix], chain_M[b_ix])
                score_val = score_list[index_seq]

                # Calculamos el recovery rate de la secuencia
                mask_for_loss = mask * chain_M * chain_M_pos
                seq_recovery_rate = torch.sum(
                    torch.sum(torch.nn.functional.one_hot(S[b_ix], 21) * torch.nn.functional.one_hot(torch.tensor(S_sample_list[index_seq][b_ix], device=device), 21), axis=-1) * mask_for_loss[b_ix]
                ) / torch.sum(mask_for_loss[b_ix])

                # Formateamos los metadatos de la cabecera del formato FASTA (sin global_val que no está definido)
                header = (f">Sample_{index_seq + 1}_Temp_{temp}_{tag_fijos}"
                f"_Score_{score_val:.4f}"
                f"_Recovery_{seq_recovery_rate.item():.4f}")
                # Escribimos cabecera y secuencia en el archivo FASTA
                fasta_file.write(f"{header}\n{seq_string}\n")
                index_seq += 1

print(f"¡Completado! Se han guardado {index_seq} secuencias en el archivo FASTA: {fasta_filename}")

¡Completado! Se han guardado 100 secuencias en el archivo FASTA: /content/secuencias_diseñadas.fasta


In [ ]:
import os
import numpy as np
import torch
import copy

# 1. Definimos los parámetros principales de la exploración
temperaturas_exploracion = [0.01, 0.1, 1.0]
estrategias_fijados = [True, False]  # True = con residuos fijos, False = sin residuos fijos
num_seqs = 100

# 2. Definimos las posiciones fijas basadas en tu archivo .txt (Phe 9, Leu 12, Phe 13, Phe 40, Ser 53, Val 59, Lys 61)
# En ProteinMPNN las posiciones fijas se definen como listas base 1 por cadena.
posiciones_fijas_lista = [9, 12, 13, 40, 53, 59, 61]

for fijar_residuos in estrategias_fijados:
    for temp in temperaturas_exploracion:
        # Configurar la carpeta y diccionario de fijos según la estrategia
        if fijar_residuos:
            # Creamos el diccionario indicando que la cadena 'A' tendrá estas posiciones bloqueadas
            fixed_positions_dict = {'binder_patulin_315': {'A': posiciones_fijas_lista}}
            tag_fijos = "con_fijos"
        else:
            fixed_positions_dict = None
            tag_fijos = "sin_fijos"

        nombre_archivo_fasta = f"/content/secuencias_temp_{temp}_{tag_fijos}.fasta"
        print(f"\n>>> Iniciando generación: Temp={temp} | Estrategia={tag_fijos}...")

        # Inicializamos almacenamiento para esta corrida
        S_sample_list_run = []
        score_list_run = []
        global_list_run = []

        with torch.no_grad():
            for ix, protein in enumerate(dataset_valid):
                batch_clones = [copy.deepcopy(protein) for _ in range(batch_size)]
                X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, visible_list_list, masked_list_list, masked_chain_length_list_list, chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, tied_pos_list_of_lists_list, pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, tied_beta = tied_featurize(
                    batch_clones, device, chain_id_dict, fixed_positions_dict, omit_AA_dict, tied_positions_dict, pssm_dict, bias_by_res_dict
                )
                pssm_log_odds_mask = (pssm_log_odds_all > pssm_threshold).float()

                for j in range(num_seqs // batch_size):
                    randn_2 = torch.randn(chain_M.shape, device=X.device)
                    if tied_positions_dict is None:
                        sample_dict = model.sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), bias_by_res=bias_by_res_all)
                    else:
                        sample_dict = model.tied_sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), tied_pos=tied_pos_list_of_lists_list[0], tied_beta=tied_beta, bias_by_res=bias_by_res_all)

                    S_sample = sample_dict["S"]
                    log_probs = model(X, S_sample, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_2, use_input_decoding_order=True, decoding_order=sample_dict["decoding_order"])
                    mask_for_loss = mask*chain_M*chain_M_pos
                    scores = _scores(S_sample, log_probs, mask_for_loss)
                    global_scores = _scores(S_sample, log_probs, mask)

                    S_sample_list_run.append(S_sample.cpu().data.numpy())
                    score_list_run.append(scores.cpu().data.numpy())
                    global_list_run.append(global_scores.cpu().data.numpy())
        # Guardar resultados de esta combinación en su propio archivo FASTA
        with open(nombre_archivo_fasta, "w") as fasta_file:
            index_seq = 0
            for idx_batch in range(len(S_sample_list_run)):
                for b_ix in range(batch_size):
                    seq_string = _S_to_seq(S_sample_list_run[idx_batch][b_ix], chain_M[b_ix])
                    score_val = score_list_run[idx_batch][b_ix]
                    global_val = global_list_run[idx_batch][b_ix]
                    # Calculamos el recovery rate global para cada secuencia
                    mask_for_loss = mask * chain_M * chain_M_pos
                    seq_recovery_rate = torch.sum(
                        torch.sum(torch.nn.functional.one_hot(S[b_ix], 21) * torch.nn.functional.one_hot(torch.tensor(S_sample_list_run[idx_batch][b_ix], device=device), 21), axis=-1) * mask_for_loss[b_ix]
                    ) / torch.sum(mask_for_loss[b_ix])

                    header = f">Sample_{index_seq + 1}_Temp_{temp}_{tag_fijos}_Score_{score_val:.4f}_Recovery_{seq_recovery_rate.item():.4f}"
                    fasta_file.write(f"{header}\n{seq_string}\n")
                    index_seq += 1

        print(f"--> ¡Guardado exitoso! 100 secuencias escritas en: {nombre_archivo_fasta}")


>>> Iniciando generación: Temp=0.01 | Estrategia=con_fijos...


In [ ]:
import copy, torch, numpy as np

temperaturas_exploracion = [0.01, 0.1, 1.0]
estrategias_fijados = [True, False]   # True = con residuos fijos
num_seqs = 100
posiciones_fijas_lista = [9, 12, 13, 40, 53, 59, 61]   # de fixed_residues_binder_315.txt

tied_positions_dict = None            # una sola cadena: no hace falta atar posiciones
nombre_pdb = pdb_dict_list[0]['name'] # 'binder_patulin_315'

torch.manual_seed(37)
np.random.seed(37)

for fijar_residuos in estrategias_fijados:
    for temp in temperaturas_exploracion:
        if fijar_residuos:
            fixed_positions_dict = {nombre_pdb: {'A': posiciones_fijas_lista}}
            tag_fijos = "con_fijos"
        else:
            fixed_positions_dict = None
            tag_fijos = "sin_fijos"

        ruta = f"/content/secuencias_temp_{temp}_{tag_fijos}.fasta"
        print(f">>> Temp={temp} | {tag_fijos}")

        with torch.no_grad(), open(ruta, "w") as fasta_file:
            batch_clones = [copy.deepcopy(dataset_valid[0])]
            (X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list,
             visible_list_list, masked_list_list, masked_chain_length_list_list,
             chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask,
             tied_pos_list_of_lists_list, pssm_coef, pssm_bias, pssm_log_odds_all,
             bias_by_res_all, tied_beta) = tied_featurize(
                batch_clones, device, chain_id_dict, fixed_positions_dict,
                omit_AA_dict, tied_positions_dict, pssm_dict, bias_by_res_dict)

            pssm_log_odds_mask = (pssm_log_odds_all > pssm_threshold).float()
            mask_for_loss = mask * chain_M * chain_M_pos   # solo posiciones rediseñadas

            for i in range(num_seqs):
                randn_2 = torch.randn(chain_M.shape, device=X.device)
                sample_dict = model.sample(
                    X, randn_2, S, chain_M, chain_encoding_all, residue_idx,
                    mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np,
                    bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos,
                    omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef,
                    pssm_bias=pssm_bias, pssm_multi=pssm_multi,
                    pssm_log_odds_flag=bool(pssm_log_odds_flag),
                    pssm_log_odds_mask=pssm_log_odds_mask,
                    pssm_bias_flag=bool(pssm_bias_flag), bias_by_res=bias_by_res_all)
                S_sample = sample_dict["S"]

                log_probs = model(X, S_sample, mask, chain_M * chain_M_pos,
                                  residue_idx, chain_encoding_all, randn_2,
                                  use_input_decoding_order=True,
                                  decoding_order=sample_dict["decoding_order"])

                score = _scores(S_sample, log_probs, mask_for_loss).item()   # solo rediseñados
                global_score = _scores(S_sample, log_probs, mask).item()     # TODA la estructura
                rec = (torch.sum(torch.sum(F.one_hot(S[0], 21) * F.one_hot(S_sample[0], 21),
                                           dim=-1) * mask_for_loss[0])
                       / torch.sum(mask_for_loss[0])).item()
                seq = _S_to_seq(S_sample[0], chain_M[0])

                header = (f">Sample_{i + 1}_Temp_{temp}_{tag_fijos}"
                          f"_Score_{score:.4f}_Global_{global_score:.4f}_Recovery_{rec:.4f}")
                fasta_file.write(f"{header}\n{seq}\n")

        print(f"--> 100 secuencias en {ruta}")

### 4. Selección y comparación mediante global_score

In [ ]:
import re, glob
import pandas as pd
from Bio import SeqIO

patron = re.compile(
    r"^(?P<id>Sample_\d+_Temp_(?P<T>[0-9.]+)_(?P<tag>con_fijos|sin_fijos))"
    r"_Score_(?P<score>[-0-9.eE]+)_Global_(?P<gs>[-0-9.eE]+)_Recovery_(?P<rec>[-0-9.eE]+)$"
)

todo, seleccion = [], []
archivos = sorted(glob.glob("/content/secuencias_temp_*_*fijos.fasta"))
assert len(archivos) == 6, f"Esperaba 6 archivos, encontré {len(archivos)}"

for ruta in archivos:
    filas = []
    for r in SeqIO.parse(ruta, "fasta"):
        m = patron.match(r.description)
        assert m, f"Encabezado inesperado: {r.description}"
        filas.append({
            "Identificador": m["id"],
            "Secuencia": str(r.seq),
            "Global_score": float(m["gs"]),
            "T_de_generación": float(m["T"]),
            "residuos_fijados": "Sí" if m["tag"] == "con_fijos" else "No",
        })
    d = pd.DataFrame(filas)
    assert len(d) == 100, f"{ruta}: {len(d)} secuencias"
    todo.append(d)
    seleccion += [d.nsmallest(3, "Global_score"), d.nlargest(3, "Global_score")]

final = pd.concat(seleccion, ignore_index=True)
assert len(final) == 36 and list(final.columns) == [
    "Identificador", "Secuencia", "Global_score", "T_de_generación", "residuos_fijados"]

final.to_csv("global_score_top_bottom.csv", index=False)
print(final.groupby(["T_de_generación", "residuos_fijados"]).size())  # 6 por grupo
final

### 5. Comparación de los valores de global_score

Construye dos gráficas tipo boxplot para comparar la distribución de global_score de las 100 secuencias
generadas en cada temperatura:

*   Un boxplot para las tres temperaturas cuando los residuos de unión se mantuvieron fijados.

*   Un boxplot para las tres temperaturas cuando los residuos de unión no se fijaron.


Cada gráfica debe mostrar las tres temperaturas (0.01, 0.1 y 1) y permitir comparar la distribución de los scores
entre ellas. Incluye títulos y etiquetas de los ejes.

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO

archivos_fasta = sorted(glob.glob("/content/secuencias_temp_*_*fijos.fasta"))
datos_plot = []

for ruta in archivos_fasta:
    for registro in SeqIO.parse(ruta, "fasta"):
        descripcion = registro.description
        try:
            partes = descripcion.split("_")

            # Buscar temperatura de forma segura
            temp_val = None
            for i, p in enumerate(partes):
                if p == "Temp" and i + 1 < len(partes):
                    temp_val = float(partes[i + 1])
                    break

            # Buscar score (usando Global si está, o Score como alternativa principal)
            score_val = None
            if "Global" in partes:
                idx = partes.index("Global") + 1
                score_val = float(partes[idx])
            elif "Score" in partes:
                idx = partes.index("Score") + 1
                score_val = float(partes[idx])

            estrategia = "Con residuos fijos" if "con_fijos" in descripcion else "Sin residuos fijos"

            if temp_val is not None and score_val is not None:
                datos_plot.append({
                    "Temperatura": temp_val,
                    "Estrategia": estrategia,
                    "Global_score": score_val
                })
        except Exception as e:
            continue

if not datos_plot:
    print("Error: No se pudieron extraer datos de los archivos FASTA. Por favor revisa el formato de las cabeceras.")
else:
    df_plot = pd.DataFrame(datos_plot)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    # 1. Boxplot con residuos fijados
    df_fijos = df_plot[df_plot["Estrategia"] == "Con residuos fijos"]
    sns.boxplot(
        data=df_fijos,
        x="Temperatura",
        y="Global_score",
        ax=axes[0],
        palette="flare",
        hue="Temperatura",
        legend=False
    )
    axes[0].set_title("Distribución de Global Score\n(Residuos de Unión Fijados)", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("Temperatura de Muestreo (T)", fontsize=11)
    axes[0].set_ylabel("Global Score (ProteinMPNN)", fontsize=11)
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)

    # 2. Boxplot sin residuos fijados
    df_sin_fijos = df_plot[df_plot["Estrategia"] == "Sin residuos fijos"]
    sns.boxplot(
        data=df_sin_fijos,
        x="Temperatura",
        y="Global_score",
        ax=axes[1],
        palette="crest",
        hue="Temperatura",
        legend=False
    )
    axes[1].set_title("Distribución de Global Score\n(Residuos de Unión No Fijados)", fontsize=13, fontweight='bold')
    axes[1].set_xlabel("Temperatura de Muestreo (T)", fontsize=11)
    axes[1].set_ylabel("", fontsize=11)
    axes[1].grid(axis='y', linestyle='--', alpha=0.7)

    plt.suptitle("Comparación del Impacto de la Temperatura en el Global Score", fontsize=15, y=1.02, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 6. Control de calidad e interpretación

Antes de finalizar el análisis, verifica lo siguiente:

*   Que cada uno de los seis archivos .fasta contenga exactamente 100 secuencias.

* Que los residuos fijados conserven su identidad aminoacídica original en las 100 secuencias correspondientes.

* Que las secuencias y sus respectivos global_score estén correctamente asociados.

* Que las gráficas incluyan los 100 valores de cada condición y no solamente las secuencias seleccionadas.


In [ ]:
import glob, re
import numpy as np
import pandas as pd
from Bio import SeqIO

patron = re.compile(
    r"^(?P<id>Sample_\d+_Temp_(?P<T>[0-9.]+)_(?P<tag>con_fijos|sin_fijos))"
    r"_Score_(?P<score>[-0-9.eE]+)_Global_(?P<gs>[-0-9.eE]+)_Recovery_(?P<rec>[-0-9.eE]+)$"
)
fijos = {9: 'F', 12: 'L', 13: 'F', 40: 'F', 53: 'S', 59: 'V', 61: 'K'}

archivos = sorted(glob.glob("/content/secuencias_temp_*_*fijos.fasta"))
assert len(archivos) == 6, f"Esperaba 6 archivos, encontré {len(archivos)}"

indice, filas = {}, []
ok_conteo = ok_fijos = ok_gs = True
print("=== CONTROL DE CALIDAD ===\n")

for ruta in archivos:
    nombre = ruta.split("/")[-1]
    recs = list(SeqIO.parse(ruta, "fasta"))
    ms = [patron.match(r.description) for r in recs]
    assert all(ms), f"{nombre}: hay encabezados sin Global (¿corriste la celda vieja?)"
    con_fijos = "con_fijos" in nombre

    n_ok = len(recs) == 100
    fix_ok = all(str(r.seq)[p - 1] == aa for r in recs for p, aa in fijos.items()) if con_fijos else None
    # Sin fijos: Score == Global (se rediseña todo). Con fijos: Global != Score.
    iguales = [abs(float(m["score"]) - float(m["gs"])) < 1e-4 for m in ms]
    gs_ok = all(iguales) if not con_fijos else not all(iguales)

    ok_conteo &= n_ok
    ok_fijos &= (fix_ok is not False)
    ok_gs &= gs_ok
    print(f"{nombre}: n={len(recs)} {'OK' if n_ok else 'ERROR'} | "
          f"fijos {'N/A' if fix_ok is None else ('OK' if fix_ok else 'ERROR')} | "
          f"Score==Global en {sum(iguales)}/100 {'OK' if gs_ok else 'ERROR'}")

    for r, m in zip(recs, ms):
        indice[m["id"]] = (str(r.seq), float(m["gs"]))
        filas.append({"id": m["id"], "T": m["T"], "tag": m["tag"], "Global_score": float(m["gs"])})

todo_df = pd.DataFrame(filas)

# --- Asociación secuencia <-> global_score (CSV contra FASTA) ---
csv = pd.read_csv("global_score_top_bottom.csv")
ok_asoc = len(csv) == 36 and all(
    i in indice and indice[i][0] == s and abs(indice[i][1] - g) < 1e-9
    for i, s, g in zip(csv["Identificador"], csv["Secuencia"], csv["Global_score"])
)

# --- Que el CSV tenga justo las 3 menores y 3 mayores de cada archivo ---
esperados = set()
for _, g in todo_df.groupby(["T", "tag"]):
    esperados |= set(g.nsmallest(3, "Global_score")["id"]) | set(g.nlargest(3, "Global_score")["id"])
ok_sel = set(csv["Identificador"]) == esperados

# --- Que las gráficas usen los 600 valores y sean Global (no Score) ---
if "df_plot" in globals():
    ok_plot = len(df_plot) == 600 and np.allclose(
        np.sort(df_plot["Global_score"].values), np.sort(todo_df["Global_score"].values), atol=1e-9)
else:
    ok_plot = False

print("\n--- RESUMEN ---")
print(f"¿Cada archivo tiene exactamente 100 secuencias?: {'SÍ' if ok_conteo else 'NO'}")
print(f"¿Los residuos fijados se conservan en con_fijos?: {'SÍ' if ok_fijos else 'NO'}")
print(f"¿Global_score coherente con Score (igual sin fijos, distinto con fijos)?: {'SÍ' if ok_gs else 'NO'}")
print(f"¿Secuencias y global_score del CSV coinciden con el FASTA?: {'SÍ' if ok_asoc else 'NO'}")
print(f"¿El CSV tiene las 3 menores y 3 mayores de cada archivo?: {'SÍ' if ok_sel else 'NO'}")
print(f"¿Los boxplots usan los 600 valores de Global_score?: {'SÍ' if ok_plot else 'NO (¿corriste la celda 25?)'}")

Finalmente, piensa en lo siguiente (que lo tendrás que contar en tu presentación):


1. ¿Cómo cambia la distribución de global_score al modificar la temperatura de muestreo?

Podemos definir el global score como promedio de la -(log-probabilidad) de la secuencia completa condicionado al esqueleto, es decir, qué tan probable le parece al modelo esa secuencia dado ese backbone, por lo que un valor más bajo indica mayor compatibilidad secuencia estructura acorde al modelo

A temperaturas bajas el modelo elige casi siempre el aminoácido más probable en cada posición, lo que produce secuencias muy similares entre sí, bajando puntuación. A T = 1 se muestrean con mayor frecuencia aminoácidos de menor probabilidad, lo que aumenta la diversidad sin embargo reduce la compatibilidad con el esqueleto acorde al modelo.

Cabe señalar que un global_score bajo refleja que la secuencia es probable para ProteinMPNN, por lo que no garantiza por sí mismo que el binder sea funcional ni que conserve su afinidad por la patulina, ya que el modelo no considera al ligando.

Además, en todas las temperaturas la condición con residuos fijados presenta un global_score mayor que la condición sin fijar, ya que los siete residuos impuestos no necesariamente son los que el modelo elegiría libremente para ese esqueleto y, al estar incluidos en el promedio global, fuerzan al modelo y elevan el valor, a cambio de esto se conservan los contactos con el ligando definidos durante el diseño.

2. ¿Qué diferencias observas entre mantener fijados los residuos de interacción y permitir que ProteinMPNN los modifique?


Fijar los residuos restringe las posiciones F9, L12, F13, F40, S53, V59 y K61. Los cuales los colocó RFdiffusion para contactar al ligando, no porque el modelo los prefiera para estabilizar el esqueleto. Dado que se está forzando a que la proteina los mantenga y no escoge el que mejor estabiliza, tienden a tener un global score más bajo.

Al no fijar los residuos,  no forzamos al modelo a que mantenga los residuos, por lo que ProteinMPNN elige libremente los 93 residuos y le damos la libertad de que escoja la secuencia más probable posible, manteniendo un global score más elevado

3. ¿Qué combinación de temperatura y condición de fijación parece producir las secuencias mejor favorecidas por ProteinMPNN?

La mejor combinación son temperaturas bajas 0.01 y 0.1 con residuos fijados.

- Temperatura baja: Reduce el ruido y prioriza residuos de alta probabilidad.   

- Residuos fijados : Conserva la identidad de los residuos de interacción y evita cambios no deseados en posiciones sensibles.

4. ¿Por qué una puntuación favorable de ProteinMPNN no garantiza que el binder conserve o mejore su capacidad de unión a la patulina?

Por que la estructura de la patulina, compuesto orgánico, depende de las interacciones químicas específicas, por lo que aunque un global_score sea bajo y aparentenement favorable, no garantiza que conservará o mejorará su afinidad. Por lo mismo es que aunque el global_Score con temperaturas 0.01 y 0.1 y con residuos no fijados sean más bajos que los fijados en esas mismas condiciones de temperatura, estos últimos son más confiables ya que modela mejor las interacciones para sus interacciones ligando-proteinas.

**Nota:** Relación patulina-pecttao liasa


Las pectato liasas y la patulina cumplen un rol importante dentro de hongos patógenos. Ambos actúan como factores de virulencia clave secretados por hongos patógenos, como Penicillium expansum. Aunque no reaccionan químicamente entre sí de forma directa, ambas forman parte de los mecanismos que ulitizan los patógenos para invadir, ablandar y colonizar el tejido vegetal:

Retomando a las pectato liasas, estas son enzimas secretadas por hongos patógenos que rompen los enlaces glucosídicos de la pectina. Su función es provocar la maceración y pudrición blanda del fruto, destruyendo la barrera física para que el hongo avance y obtenga nutrientes.

Mientras que la patulina es una toxina producida durante la colonización . Esta contribuye a la virulencia induciendo estrés oxidativo, alterando la homeostasis celular y contribuyendo a la muerte celular.

La producción tanto de las pectato liasas como de la patulina está controlada por las mismas redes de señalización y factores ambientales del hongo durante el ataque al fruto. Factores como el PH activan factores de transcripción que regulan simultáneamente los genes que codifican para las pectato liasas y la patulina.

Finalmente, en la agroindustria, exite otro tipo de relación ya que la acción natural de las pectato liasas fúngicas indica deterioro del fruto y suele ir acompañada de altos niveles de patulina en el tejido dañado.
